# REPSOL Model Training (70/15/15)

**ResNet-50**

ResNet-50 pretrained on ImageNet (IMAGENET1K_V2 weights). Widely used in industrial sound classification research as a spectrogram-based feature extractor. Deeper residual architecture than EfficientNet-B0 (~25M vs 5M parameters), with a Dropout(0.3) layer before the classification head to reduce overfitting on the small REPSOL dataset.

This notebook runs the full pipeline:
1. Configure training hyperparameters.
2. Verify spectrogram tensor files in train/val/test.
3. Train ResNet-50 with live progress bars.
4. Evaluate on validation and test sets.
5. Print detailed classification report and confusion matrix.

In [1]:
from pathlib import Path
import torch

# ===== Hyperparameters (edit these) =====
BATCH_SIZE    = 8       # keep small for CPU; increase to 32+ on GPU
EPOCHS        = 15
LEARNING_RATE = 1e-4   # lower than EfficientNet: ResNet-50 is larger, finer tuning needed
PATIENCE      = 4
MODEL_NAME    = "resnet50"

# ===== Paths =====
PROJECT_ROOT   = Path(r"D:\Work\Internships\INMAR\REPSOL")
SPECTROGRAM_DIR = PROJECT_ROOT / "Data" / "Spectrograms"
OUTPUT_DIR     = PROJECT_ROOT / "Models_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def next_run_path(model_name: str, suffix: str, ext: str, output_dir: Path) -> Path:
    prefix = f"{model_name}{suffix}"
    existing_numbers = [0]
    for path in output_dir.iterdir():
        if not path.is_file() or path.suffix != ext:
            continue
        stem = path.stem
        if stem == prefix:
            existing_numbers.append(0)
            continue
        if stem.startswith(prefix + "_"):
            suffix_text = stem[len(prefix) + 1:]
            if suffix_text.isdigit():
                existing_numbers.append(int(suffix_text))
    next_num = max(existing_numbers) + 1
    return output_dir / f"{prefix}_{next_num:02d}{ext}"


CHECKPOINT_PATH = next_run_path(MODEL_NAME, "_best", ".pth", OUTPUT_DIR)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("SPECTROGRAM_DIR:", SPECTROGRAM_DIR)
print("OUTPUT_DIR     :", OUTPUT_DIR)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("DEVICE         :", DEVICE)
print(f"BATCH_SIZE: {BATCH_SIZE}  |  EPOCHS: {EPOCHS}  |  LR: {LEARNING_RATE}")

PROJECT_ROOT   : D:\Work\Internships\INMAR\REPSOL
SPECTROGRAM_DIR: D:\Work\Internships\INMAR\REPSOL\Data\Spectrograms
OUTPUT_DIR     : D:\Work\Internships\INMAR\REPSOL\Models_output
CHECKPOINT_PATH: D:\Work\Internships\INMAR\REPSOL\Models_output\resnet50_best_02.pth
DEVICE         : cpu
BATCH_SIZE: 8  |  EPOCHS: 15  |  LR: 0.0001


In [2]:
def count_pt_files(root):
    counts = {}
    for split in ["train", "val", "test"]:
        split_dir = root / split
        counts[split] = sum(1 for _ in split_dir.rglob("*.norm.pt")) if split_dir.exists() else 0
    return counts

counts = count_pt_files(SPECTROGRAM_DIR)
print("Tensor files by split:", counts)
print("Total:", sum(counts.values()))

assert counts["train"] > 0, "No train .norm.pt files found."
assert counts["val"]   > 0, "No val .norm.pt files found."
assert counts["test"]  > 0, "No test .norm.pt files found."

Tensor files by split: {'train': 1382, 'val': 296, 'test': 297}
Total: 1975


In [3]:
import importlib
import subprocess
import sys

required = ["torch", "torchvision", "torchaudio", "scikit-learn", "pandas", "tqdm", "numpy"]
name_map = {"scikit-learn": "sklearn"}

for pkg in required:
    import_name = name_map.get(pkg, pkg.replace("-", "_"))
    try:
        importlib.import_module(import_name)
        print(f"OK: {pkg}")
    except Exception:
        print(f"Installing: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"Installed: {pkg}")

OK: torch
OK: torchvision
OK: torchaudio
OK: scikit-learn
OK: pandas
OK: tqdm
OK: numpy


---

## Training

In [4]:
import importlib
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import src.ResNet.model as model_module
import src.ResNet.train as train_module

model_module = importlib.reload(model_module)
train_module = importlib.reload(train_module)
Trainer = train_module.Trainer

trainer = Trainer(
    spectrogram_dir=SPECTROGRAM_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    max_epochs=EPOCHS,
    patience=PATIENCE,
    lr=LEARNING_RATE,
    device=DEVICE,
)

trainer.fit()

Train samples: 1382
Train batches: 173


Epoch 1/15 Validation: 100%|████████████████████████| 37/37 [13:21<00:00, 21.65s/batch, loss=2.0837]

Epoch 1/15 | Train Loss: 1.6787 | Val Loss: 1.2328 | Train Acc: 38.42 | Val Acc: 61.15


Saved improved checkpoint to: D:\Work\Internships\INMAR\REPSOL\Models_output\resnet50_best_02.pth


Epoch 2/15 Validation: 100%|████████████████████████| 37/37 [09:00<00:00, 14.60s/batch, loss=2.7421]

...

Epoch 6/15 | Train Loss: 0.4869 | Val Loss: 1.1812 | Train Acc: 81.26 | Val Acc: 57.77
No improvement for 2/4 epochs



Epoch 7/15 Validation: 100%|████████████████████████| 37/37 [08:25<00:00, 13.66s/batch, loss=3.4792]

Epoch 7/15 | Train Loss: 0.3261 | Val Loss: 1.3960 | Train Acc: 86.69 | Val Acc: 62.84
No improvement for 3/4 epochs



Epoch 8/15 Validation: 100%|████████████████████████| 37/37 [08:22<00:00, 13.57s/batch, loss=1.0978]

Epoch 8/15 | Train Loss: 0.1672 | Val Loss: 1.1909 | Train Acc: 93.05 | Val Acc: 63.51
No improvement for 4/4 epochs
Early stopping: no improvement for 4 epochs.
Training finished.


---

## Evaluation

In [5]:
import importlib
import torch
import src.evaluate as eval_module

eval_module = importlib.reload(eval_module)
evaluate_model = eval_module.evaluate_model

state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
trainer.model.load_state_dict(state)

val_metrics  = evaluate_model(trainer.model, trainer.val_loader,  DEVICE)
test_metrics = evaluate_model(trainer.model, trainer.test_loader, DEVICE)

print("Validation Metrics")
print({k: round(val_metrics[k], 4) for k in ["accuracy", "precision", "recall", "f1"]})

print("\nTest Metrics")
print({k: round(test_metrics[k], 4) for k in ["accuracy", "precision", "recall", "f1"]})

Validation Metrics
{'accuracy': 0.5912, 'precision': 0.6699, 'recall': 0.5912, 'f1': 0.5776}

Test Metrics
{'accuracy': 0.6128, 'precision': 0.7033, 'recall': 0.6128, 'f1': 0.6031}


In [6]:
print("Test Classification Report:\n")
print(test_metrics["report"])

print("Test Confusion Matrix:")
print(test_metrics["confusion_matrix"])

Test Classification Report:

              precision    recall  f1-score   support

           0       0.52      0.79      0.63        33
           1       0.67      0.33      0.44         6
           2       0.44      0.95      0.60        20
           3       0.88      0.35      0.50       106
           4       0.44      0.64      0.52        39
           5       0.50      0.73      0.59        11
           6       0.81      0.83      0.82        76
           7       0.25      0.33      0.29         6

    accuracy                           0.61       297
   macro avg       0.56      0.62      0.55       297
weighted avg       0.70      0.61      0.60       297

Test Confusion Matrix:
[[26  0  0  0  5  0  2  0]
 [ 0  2  0  2  0  2  0  0]
 [ 0  0 19  0  1  0  0  0]
 [16  1 15 37 24  1  9  3]
 [ 2  0  9  2 25  0  1  0]
 [ 0  0  0  0  0  8  0  3]
 [ 6  0  0  1  2  4 63  0]
 [ 0  0  0  0  0  1  3  2]]
